# 🎯 AI Career Skill Gap Analyzer — Experiment Notebook

This notebook walks through the full pipeline interactively:
1. Load & explore the jobs dataset
2. NLP preprocessing
3. Skill extraction
4. TF-IDF model training
5. Skill gap analysis
6. Job recommendations
7. Visualisations

In [ ]:
# Setup – run from project root
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports OK')

## 1. Load & Explore Dataset

In [ ]:
from utils.similarity import load_dataset

df = load_dataset('../data/jobs_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Skill count per role
df['skill_count'] = df['skills_list'].apply(len)

plt.figure(figsize=(12, 5))
plt.barh(df['job_role'], df['skill_count'], color='steelblue')
plt.xlabel('Number of Required Skills')
plt.title('Required Skills per Job Role')
plt.tight_layout()
plt.show()

In [ ]:
# Most common skills across all roles
from collections import Counter

all_skills = [s for lst in df['skills_list'] for s in lst]
skill_freq = Counter(all_skills).most_common(20)

skills_df = pd.DataFrame(skill_freq, columns=['skill', 'count'])

plt.figure(figsize=(10, 6))
plt.barh(skills_df['skill'][::-1], skills_df['count'][::-1], color='coral')
plt.xlabel('Frequency')
plt.title('Top 20 Most In-Demand Skills')
plt.tight_layout()
plt.show()

## 2. NLP Preprocessing

In [ ]:
from utils.text_preprocessing import clean_text, tokenize, remove_stopwords, lemmatize, preprocess

sample_resume = """
Senior Data Scientist with 5 years of experience in Machine Learning and AI.
I have built recommendation systems using Python, TensorFlow, and PyTorch.
Strong background in SQL, PostgreSQL, and data warehousing.
Deployed models with Docker and Kubernetes on AWS.
Contact: john.doe@email.com | github.com/johndoe
"""

print('--- RAW ---')
print(sample_resume[:200])

print('\n--- CLEANED ---')
print(clean_text(sample_resume))

print('\n--- TOKENS (first 15) ---')
tokens = tokenize(clean_text(sample_resume))
print(tokens[:15])

print('\n--- AFTER STOPWORD REMOVAL (first 15) ---')
filtered = remove_stopwords(tokens)
print(filtered[:15])

print('\n--- AFTER LEMMATIZATION (first 15) ---')
lemmas = lemmatize(filtered)
print(lemmas[:15])

## 3. Skill Extraction

In [ ]:
from utils.skill_extraction import extract_skills

skills = extract_skills(sample_resume)
print(f'Found {len(skills)} skills:')
for s in skills:
    print(f'  • {s}')

## 4. Train TF-IDF Model

In [ ]:
from utils.similarity import train_tfidf

vectorizer, job_matrix = train_tfidf(df, save=True)
print(f'Vocab size: {len(vectorizer.vocabulary_)}')
print(f'Job matrix shape: {job_matrix.shape}')

## 5. Skill Gap Analysis

In [ ]:
from utils.similarity import analyze_skill_gap

my_skills = ['python', 'pandas', 'numpy', 'machine learning', 'sql', 'git']
target_role_skills = ['python', 'machine learning', 'tensorflow', 'statistics',
                       'pandas', 'numpy', 'scikit-learn', 'sql', 'deep learning',
                       'data visualization']

gap = analyze_skill_gap(my_skills, target_role_skills)
print(f"Match Score: {gap['match_score']}%")
print(f"Matched ({gap['total_matched']}): {gap['matched_skills']}")
print(f"Missing ({len(gap['missing_skills'])}): {gap['missing_skills']}")

## 6. Job Recommendations

In [ ]:
from utils.similarity import recommend_jobs_tfidf

my_skills = ['python', 'tensorflow', 'pytorch', 'nlp', 'machine learning',
             'pandas', 'numpy', 'scikit-learn', 'sql', 'git', 'docker']

recs = recommend_jobs_tfidf(my_skills, top_n=5)

print('🎯 TOP JOB RECOMMENDATIONS\n')
for i, r in enumerate(recs, 1):
    print(f'{i}. {r["job_role"]}')
    print(f'   Similarity: {r["similarity_score"]:.2%}  |  Match: {r["match_score_pct"]}%')
    print(f'   ✔ Have: {r["matched_skills"]}')
    print(f'   ✖ Need: {r["missing_skills"]}')
    print()

In [ ]:
# Visualize match scores
roles = [r['job_role'] for r in recs]
scores = [r['match_score_pct'] for r in recs]

plt.figure(figsize=(10, 5))
bars = plt.barh(roles[::-1], scores[::-1], color='mediumseagreen')
plt.xlabel('Match Score (%)')
plt.title('Resume → Job Role Match Scores')
for bar, score in zip(bars, scores[::-1]):
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{score}%', va='center')
plt.tight_layout()
plt.show()

## 7. End-to-End: Full Resume Analysis

In [ ]:
from utils.similarity import analyze_resume
import json

result = analyze_resume(sample_resume, top_n=5)
print(json.dumps(result, indent=2))